In [2]:
import pandas as pd
import tushare as ts
pro = ts.pro_api('b7bf12884a6ba2a5b33717fea342a6e8d3ab2fa80b5bb8ff6682e20f')
df = pro.index_weight(index_code='000300.SH', start_date='20230101', end_date='20251231')
df

,index_code,con_code,trade_date,weight
0,000300.SH,300750.SZ,20251231,3.8440
1,000300.SH,600519.SH,20251231,3.4130
2,000300.SH,601318.SH,20251231,2.8860
3,000300.SH,300308.SZ,20251231,2.6820
4,000300.SH,601899.SH,20251231,2.2470
...,...,...,...,...
5995,000300.SH,688082.SH,20250303,0.0455
5996,000300.SH,601808.SH,20250303,0.0415
5997,000300.SH,000800.SZ,20250303,0.0385
5998,000300.SH,600377.SH,20250303,0.0373


In [4]:
import numpy as np
import time
hs300_code_list = df['con_code'].drop_duplicates().to_list()
dict_stock = {}
for code in hs300_code_list:
    hs300_daily = pro.daily(ts_code=str(code), start_date='20230101', end_date='20251231', fields='ts_code,trade_date,open,close')
    hs300_daily['ma5'] = hs300_daily['close'].rolling(window=5).mean()
    hs300_daily['ma20'] = hs300_daily['close'].rolling(window=20).mean()
    signal = pd.Series(np.where((hs300_daily['ma5'] - hs300_daily['ma20'])>0, 1, 0))
    cross = signal.diff().fillna(0)
    hs300_daily['cross'] = cross
    if hs300_daily.empty:
        print('fail!')
        break
    time.sleep(0.12)
    dict_stock[code] = hs300_daily
print('dict_stock is ready!')

dict_stock is ready!


In [13]:
sample = hs300_code_list[:10]
volume = 100  # 股
time_begin = 20240101
time_end = 20241231
for code in sample:
    df = dict_stock[code]
    df = df[(df['trade_date']>=str(time_begin)) & (df['trade_date']<=str(time_end))]
    buy_price = df[df['cross']==1]['close'].values
    sell_price = df[df['cross']==-1]['close'].values
    if len(buy_price)==0 or len(sell_price)==0:
        yearly_return = 0
    if len(buy_price) > len(sell_price):
        buy_price = buy_price[:len(sell_price)]
    elif len(buy_price) < len(sell_price):
        sell_price = sell_price[:len(buy_price)]
    yearly_return = (sell_price - buy_price).sum()*volume
    yearly_return_ratio = yearly_return / (buy_price.sum()*volume)
    print(f'{code} 年化收益: {yearly_return}元, 年化收益率: {yearly_return_ratio}%')
print('over')

300750.SZ 年化收益: -9108.000000000004元, 年化收益率: -0.05294302289082393%
600519.SH 年化收益: -31242.000000000007元, 年化收益率: -0.02215408441555679%
601318.SH 年化收益: 707.0元, 年化收益率: 0.020867768595041324%
300308.SZ 年化收益: -7200.000000000002元, 年化收益率: -0.07258576714082648%
601899.SH 年化收益: -113.99999999999989元, 年化收益率: -0.008510638297872334%
600036.SH 年化收益: -147.9999999999997元, 年化收益率: -0.00709117914810022%
300502.SZ 年化收益: -5584.999999999998元, 年化收益率: -0.0804175665946724%
000333.SZ 年化收益: 404.00000000000136元, 年化收益率: 0.010212593847165028%
601166.SH 年化收益: -282.99999999999983元, 年化收益率: -0.02307566862361381%
600900.SH 年化收益: 341.99999999999943元, 年化收益率: 0.018244865297412614%
over
